## Using MAFFT for Multiple Sequence Alignment (MSA)

MAFFT is a tool that aligns many sequences at once to accurately determine evolutionary similarities between the sequences.

For this mini project, I'm going to use a FASTA file with the CDS of TP53 (the gene for p53) of human orthologs.

In [1]:
from Bio import SeqIO # Used to read and write biological sequence data (like FASTA files)
import subprocess # Allows Python to run external programs like MAFFT
from io import StringIO # Lets you treat a string like a file (needed for parsing text as if it came from a file)
from Bio.SeqRecord import SeqRecord # Data structure from Biopython that holds .seq, .id, and .description

sequence_list = list(SeqIO.parse('../sequence-data/Human_TP53_orthologues.fa', 'fasta')) # Fastest way to parse fasta file and sort each sequence and its metadata as a Bio.SeqRecord.SeqRecord object
print('Number of human orthologs for TP53:', len(sequence_list))

# It's more common to do MSA on amino acid sequences, so I'm going to translate each CDS:
translated_records = []
for seq in sequence_list:
    protein = seq.seq.translate(to_stop=True) # to_stop=True means to stop at the first stop codon and to not put the * at the end
    record = SeqRecord(protein, id=seq.id, description=seq.description)
    translated_records.append(record)

# Confirms successful translation
print(translated_records[0].seq)
print()
print(sequence_list[0].seq.translate(to_stop=True))

Number of human orthologs for TP53: 253
MEEETFSLPLSQDTFQDLWENVAAPSISTIQTTVSGNECWQDGSLTMALMDMPYDEDLFNLPSELPNKDGANSSCPTVPVTTDHPGEYDFKLRFQKSGTAKSVTSTYSESLNKLYCQLAKTSPLEVLLSREPPLGAMLRATAIYKKTEHVAEVVRRCPHHQNEDSTENRSHLIRMEGSQRAQYFEDPHTKRQSVTVPYEPPQLGSEFTTILLSFMCNSSCMGGMNRRPILAILTLETQEGVVLGRRCFEVRVCACPGRDRKTEEANSTKMQTETKDAKKRKSAPTSDSTTVKKSRTASSAEEDDKEVFTLQIRGRKRYEMIKRINDGLDLLENKTKSKTTYKPEGPVLPSGKRLMHRGEKSDSD

MEEETFSLPLSQDTFQDLWENVAAPSISTIQTTVSGNECWQDGSLTMALMDMPYDEDLFNLPSELPNKDGANSSCPTVPVTTDHPGEYDFKLRFQKSGTAKSVTSTYSESLNKLYCQLAKTSPLEVLLSREPPLGAMLRATAIYKKTEHVAEVVRRCPHHQNEDSTENRSHLIRMEGSQRAQYFEDPHTKRQSVTVPYEPPQLGSEFTTILLSFMCNSSCMGGMNRRPILAILTLETQEGVVLGRRCFEVRVCACPGRDRKTEEANSTKMQTETKDAKKRKSAPTSDSTTVKKSRTASSAEEDDKEVFTLQIRGRKRYEMIKRINDGLDLLENKTKSKTTYKPEGPVLPSGKRLMHRGEKSDSD


You would run this command to quickly run MAFFT for MSA, as it's the equivalent of a command line command:

<code>!cat ../sequence-data/gencode.v48.pc_transcripts.fa | mafft --quiet - > aligned-sequences.fa</code>
Where...
- ! indicates a terminal command
- 'cat' means 'concatenate' to print the contents of a file
- So overall, <code>!cat ../sequence-data/gencode.v48.pc_transcripts.fa</code> just reads the FASTA file
- | is a pipe that takes the output (the FASTA file and its contents) from the previous command (cat ...) and sends it as an input to the following command (mafft ...)
- <code>mafft --quiet -</code> calls MAFFT, --quiet hides unnecessary status outputs, and - tells MAFFT to read what the pipe gives it (this is called the standard input)
- <code>> aligned-sequences.fa</code> redirects the output from the screen to a file called 'aligned-sequences.fa'

In [2]:
# The code below is a simple way of taking all of the translated_records Bio.SeqRecord.SeqRecord objects and combining them as one big string, but there are faster methods.

#seq_str = ''

#for seq in translated_records:
#    seq_str += '>' + seq.description + '\n'
#    seq_str += str(seq.seq) + '\n'

# Faster method:
lines = []

for seq in translated_records:
    lines.append('>' + seq.description)
    lines.append(str(seq.seq))

seq_str = '\n'.join(lines) + '\n'

# Fastest, simplest method:
output = StringIO()
SeqIO.write(translated_records, output, 'fasta')
seq_str = output.getvalue()

seq_str[0:391] # First sequence

'>ENSTNIP00000013035\nMEEETFSLPLSQDTFQDLWENVAAPSISTIQTTVSGNECWQDGSLTMALMDMPYDEDLFN\nLPSELPNKDGANSSCPTVPVTTDHPGEYDFKLRFQKSGTAKSVTSTYSESLNKLYCQLAK\nTSPLEVLLSREPPLGAMLRATAIYKKTEHVAEVVRRCPHHQNEDSTENRSHLIRMEGSQR\nAQYFEDPHTKRQSVTVPYEPPQLGSEFTTILLSFMCNSSCMGGMNRRPILAILTLETQEG\nVVLGRRCFEVRVCACPGRDRKTEEANSTKMQTETKDAKKRKSAPTSDSTTVKKSRTASSA\nEEDDKEVFTLQIRGRKRYEMIKRINDGLDLLENKTKSKTTYKPEGPVLPSGKRLMHRGEK\nSDSD\n'

In [3]:
# The code below does the same stuff as this command:
# !cat ../sequence-data/gencode.v48.pc_transcripts.fa | mafft --quiet - > aligned-sequences.fa
# Except no intermediate file is produced as a result

child = subprocess.Popen(       # 'Popen' means 'process open'; 'subprocess' is a Python module needed to call 'Popen()'; subprocess.Popen(...) runs the MAFFT program from inside Python
    ['mafft', '--quiet', '-'],  # Using the same arguments as the command-line arguments (see what they mean in the markdown box above)
    stdin=subprocess.PIPE,      # Sets the standard input to whatever the pipe gives -- Python will send the sequences to MAFFT through a pipe (not a file)
    stdout=subprocess.PIPE,     # Sets the standard output to MAFFT's output (the aligned sequences) -- Python will capture MAFFT's output
    stderr=subprocess.PIPE      # Sets the standard error messages to MAFFT's error messages -- Python will capture any errors from MAFFT
)

child_out, child_err = child.communicate(input=seq_str.encode())
# In the line above...
# child.communicate(...) Sends the input (the sequences in seq_str) into MAFFT, and collects child_out (the aligned sequences from MAFFT) and child_err (any error messages)
# .encode() Turns the string into bytes so it can be sent to and interpreted by MAFFT

# Parsing the result:
aligned_seqs = list(SeqIO.parse(StringIO(child_out.decode()), "fasta"))
# In the line above...
# child_out.decode() turns MAFFT's output from bytes into a string
# StringIO Makes that string behave like a file so SeqIO can read it
# SeqIO.parse(..., 'fasta') reads the aligned sequences from the MAFFT output
# list(...) turns the parser result into a list of SeqRecord objects

Other things to take note of when performing MSA with MAFFT:
- <code>--auto</code>: MAFFT chooses the best algorithm based on the data size
- <code>--localpair</code> or <code>--globalpair</code>: For slower but more accurate alignments
- <code>--maxiterate 1000</code>: Improves refinement

To use these, just add them into the <code>['mafft', '...', '...']</code> list.

In [ ]:
# MAFFT alignment results:
# It's good when the sequences' series of letters line up in certain places multiple times, because it means MAFFT likely properly aligned them
for i, aligned in enumerate(aligned_seqs):
    print(f'Aligned Protein Sequence {i + 1}')
    print(aligned.seq)
    print()

In [5]:
# This saves the alignments as a file to be analyzed and interpreted with Jalview, an external application
with open('../sequence-data/aligned_Human_TP53_orthologues.fasta', 'w') as output_handle:
    SeqIO.write(aligned_seqs, output_handle, 'fasta')

## Using MSA to Determine Phylogenies with IQ-TREE and ETE3

IQ-TREE infers the tree based on the MSA, and ETE3 can be used to visualize and analyze it.

### NOTE: DO NOT RUN THE CELL BELOW WITHOUT CAUTION OR WITHOUT CHANGING THE OUTPUT FOLDER FIRST

In [ ]:
import os

# Path to the aligned FASTA file containing protein sequences of TP53 orthologues
aligned_fasta = '../sequence-data/aligned_Human_TP53_orthologues.fasta'

# Directory where all IQ-TREE output files will be saved
output_folder = '../sequence-data/iq-tree_outputs'

# Create a prefix for all output files
iqtree_prefix = os.path.join(output_folder, 'my_iqtree_run')

# Run IQ-TREE using subprocess:
# - 'iqtree': the command-line program
# - '-s': specify the sequence alignment file
# - '-m MFP': use ModelFinder Plus to automatically select the best-fit substitution model
# - '-bb 1000': perform 1000 ultrafast bootstrap replicates to assess support for each branch
# - '-pre': set the prefix for all output files (tree, log, model info, etc.)
# - stdout= and stderr=PIPE: capture the command output (but not shown unless printed later)
subprocess.run([
    'iqtree',
    '-s', aligned_fasta,
    '-m', 'MFP', # This will make the whole process take a WHILE but will probably yield the best results. For more efficiency, just give it one or a few models to use/try.
    '-bb', '1000',
    '-pre', iqtree_prefix
    # Add -nt AUTO to make IQ-TREE use multiple CPU cores; the process will go faster
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

### IQ-TREE Output Files

The IQ-TREE analysis generates several output files, each containing specific results or intermediate data from the phylogenetic inference:

* **`my_iqtree_run.log`:** This is the screen log file that captures all the output printed to the console during the IQ-TREE run, providing a complete record of the analysis steps, progress, and any warnings.
* **`my_iqtree_run.model.gz`:** A gzipped file containing detailed information about the evolutionary model chosen by ModelFinder, including its parameters and selection criteria.
* **`my_iqtree_run.uniqueseq.phy`:** This file contains only the unique sequences from your input alignment, which IQ-TREE often uses internally to optimize computational efficiency.
* **`my_iqtree_run.ckp.gz`:** A gzipped checkpoint file. This file allows IQ-TREE to resume an interrupted analysis from the last saved state, preventing the need to restart a long run from the beginning.
* **`my_iqtree_run.treefile`:** This is your primary output file containing the best-scoring Maximum-Likelihood phylogenetic tree found by IQ-TREE, presented in Newick format. This is the tree you typically visualize and interpret for your primary phylogenetic relationships.
* **`my_iqtree_run.bionj`:** This file contains a BioNJ (Bio Neighbor-Joining) tree. BioNJ is a fast distance-based method used by IQ-TREE to generate an initial tree topology, which serves as a starting point for more complex Maximum Likelihood searches.
* **`my_iqtree_run.mldist`:** A file containing the pairwise Maximum Likelihood estimates of evolutionary distances between all sequences in your dataset, based on the selected evolutionary model.
* **`my_iqtree_run.contree`:** This file contains the bootstrap consensus tree, usually in Newick format. It summarizes the relationships consistently found across the bootstrap replicates, with the ultrafast bootstrap support values mapped onto its branches.
* **`my_iqtree_run.splits.nex`:** This NEXUS file provides detailed information about the bipartitions (splits) present in the trees and their corresponding ultrafast bootstrap support values.
* **`my_iqtree_run.iqtree`:** This is a comprehensive report of the entire IQ-TREE analysis, summarizing various statistics, details about the model, the final tree, and other important output parameters.

# Understanding Phylogenetic Inference with IQ-TREE: A Step-by-Step Walkthrough

This section details the process and functionality of IQ-TREE, a powerful software for reconstructing phylogenetic (evolutionary) trees using the Maximum Likelihood (ML) method. We will walk through the key steps involved in generating a phylogenetic tree from sequence data, referencing specific outputs from our `my_iqtree_run` analysis.

## 1. Model Selection (ModelFinder)

The very first and critical step in Maximum Likelihood phylogenetics is selecting the appropriate evolutionary model. An evolutionary model describes the probabilities of character (e.g., amino acid) changes over time. Choosing the correct model is vital for accurate tree reconstruction.

* **How it functions:** IQ-TREE's **ModelFinder** module automatically evaluates a large number of substitution models on a fast likelihood tree (an accurate enough tree made via parsimony and refined with fast likelihood) to determine which one best fits your specific sequence data. It calculates various statistical criteria for each model, such as:
    * **Akaike Information Criterion (AIC)**
    * **Corrected Akaike Information Criterion (AICc)**
    * **Bayesian Information Criterion (BIC)**
    The model with the lowest value for these criteria is generally preferred, balancing model complexity with its fit to the data. BIC is often favored for its more conservative penalty on complex models.

* **Our Analysis Results:** In our run, the best-fit model identified was **Q.MAMMAL+R7**.
    * **`Q.MAMMAL`**: This refers to an empirical protein substitution matrix specifically derived for mammalian protein sequences, describing the rates at which one amino acid is likely to change into another.
    * **`+R7`**: This indicates the inclusion of a "FreeRate model with 7 categories" (or Gamma distributed rates). This component accounts for **rate heterogeneity across sites**, meaning that different positions (sites) in a protein sequence can evolve at different speeds (e.g., highly conserved active sites evolve slowly, while surface loops might evolve quickly). The '7 categories' mean the substitution rates are divided into 7 discrete bins, each with a specific rate and proportion of sites. Our log shows these final estimated site proportions and rates, such as `(0.089, 0.053)` (0.089 proportion of sites evolving at a relative rate of 0.053).

* **Relevant Output File:**
    * `my_iqtree_run.model.gz`: This gzipped file contains all the detailed information about the model selection process and the chosen model.

## 2. Initial Tree Estimation

Before starting an intensive Maximum Likelihood search, IQ-TREE needs one or more starting phylogenetic trees. Searching through all possible tree topologies is computationally impossible for even moderately sized datasets.

* **How it functions:** IQ-TREE employs fast, heuristic methods to generate these initial trees:
    * **RapidNJ Tree:** A quick distance-based tree (Neighbor-Joining algorithm) is constructed first. This provides a very fast, albeit often not perfectly optimal, initial topology. The log shows its initial log-likelihood (e.g., `-65225.065`).
    * **Parsimony Trees:** IQ-TREE also generates multiple parsimony trees (e.g., "98 parsimony trees" in the log). Parsimony seeks the tree that requires the fewest evolutionary changes to explain the data. These provide a diverse set of starting points to ensure the subsequent ML search explores a wide range of tree shapes.
    * **Initial Model Parameter Optimization:** Before the main tree search, the model parameters (like the site proportions and rates) are initially optimized on one of these early trees to provide a good starting point for likelihood calculations. This is reflected in the "Optimal log-likelihood" lines before the main NNI search begins, and also in the generation of "Likelihood distances".

* **Relevant Output Files:**
    * `my_iqtree_run.bionj`: This file contains the initial BioNJ tree.
    * `my_iqtree_run.mldist`: This file contains the pairwise Maximum Likelihood distances calculated between all sequences based on the initial model parameters. A "WARNING: Some pairwise ML distances are too long (saturated)" might appear here if very divergent sequences make exact distance estimation challenging.

## 3. Tree Search Optimization (NNI)

This is the core iterative process where IQ-TREE searches for the best possible tree topology that maximizes the log-likelihood (i.e., the tree that best explains your sequence data given the chosen evolutionary model).

* **How it functions:** IQ-TREE uses a heuristic search algorithm called **Nearest Neighbor Interchange (NNI)**.
    * **NNI Operation:** NNI works by taking a current tree and systematically swapping adjacent subtrees around an internal branch. For each swap, it calculates the log-likelihood of the resulting new tree. If a swap leads to a higher log-likelihood, the tree is updated to this new, better topology.
    * **Iterative Refinement:** The log shows many "Optimizing NNI: done in..." lines followed by "BETTER TREE FOUND at iteration X: -LogL". This indicates that IQ-TREE is constantly performing NNI rearrangements and updating the best tree found so far. The log-likelihood value becomes less negative as the tree fit improves.
    * **Multiple Starting Points:** IQ-TREE doesn't just run NNI on one tree; it performs it on a set of the best initial trees (e.g., "Do NNI search on 20 best initial trees") to thoroughly explore the tree space and reduce the chance of getting stuck in a suboptimal "local peak."
    * **Log-likelihood Cutoff:** The "Log-likelihood cutoff on original alignment" (e.g., `-64230.298`) is a stopping criterion. If the improvements in log-likelihood fall below a certain threshold, the search may stop, assuming further iterations are unlikely to yield significantly better trees.

## 4. Ultrafast Bootstrap (UFBoot) for Branch Support

While the Maximum Likelihood tree provides the best-fitting topology, we also need to assess the statistical confidence in the relationships (branches) shown in that tree. IQ-TREE uses a highly efficient method called the Ultrafast Bootstrap (UFBoot).

* **Purpose:** Bootstrap support values (often expressed as percentages) indicate how frequently a particular clade is recovered across many resampled datasets. Higher values (e.g., 90-100%) imply strong statistical support for that clade.
* **How it functions (The "Ultrafast" Part):**
    1.  **Resampling Sites:** At the beginning of the run ("Generating 1000 samples for ultrafast bootstrap"), IQ-TREE creates 1000 new datasets (bootstrap alignments) by randomly sampling columns (sites) from your *original* alignment with replacement. **Crucially, all taxa (sequences/rows) are present in every single bootstrap replicate, but the sequence data for each taxon is slightly different due to this site resampling.**
    2.  **Efficient Evaluation:** Instead of running a full, independent Maximum Likelihood (ML) tree search for each of the 1000 bootstrap datasets (which would be extremely slow), UFBoot works in parallel with the main ML search on your original data. Each bootstrap replicate serves as a unique dataset to evaluate candidate tree topologies.
    3.  **Candidate Topology Evaluation (RELL):** Every time the main ML search finds a "BETTER TREE FOUND" on the original dataset (through NNI), it's evaluated against each bootstrap replicate. For that specific replicate, this new candidate tree's RELL (a method called **Resampling Estimated Log-Likelihood (RELL)** that quickly calculates how well that fixed candidate topology fits each resampled dataset) score is compared to the RELL scores of all other active candidate trees, including the one that was previously the "winning" candidate for that replicate.
    4.  **Clade Tallying:** For each bootstrap replicate, the candidate topology (from the set being evaluated, primarily the main ML tree's topology) that yields the highest RELL score (i.e., best fits that specific bootstrap dataset) is identified. The clades (or bipartitions) present in this "winning" topology are then recorded for that particular bootstrap replicate. This recorded clade information is then used to cumulatively tally how often each split/clade occurs across these chosen trees at specific points during the NNI optimization and at the end of the analysis. NOTE: Each time a replicate encounters a new "winning" candidate tree with a better RELL score than the last, its clade information replaces the old tree's clade information to prevent outdated information from affecting the support values. This process ensures that the support values reflect the robustness of the clades found in the main analysis.
    5.  **Convergence Monitoring:** The "NOTE: Bootstrap correlation coefficient of split occurrence frequencies" (e.g., `0.995`) indicates that IQ-TREE is monitoring the consistency of the clades found across the 1000 bootstrap evaluations. A value close to 1 means the relative frequencies of these clades (and thus the bootstrap support values) are stable and have largely converged, indicating the analysis has run long enough for reliable support estimation.

* **Relevant Output Files:**
    * `my_iqtree_run.splits.nex`: This NEXUS file contains detailed information on the splits found and their occurrence frequencies (which translate to support values).
    * `my_iqtree_run.contree`: This file provides the final bootstrap consensus tree, which is a summary tree derived from the 1000 bootstrap replicates, with the support values mapped onto its branches.

## 5. Final Optimization and Results Summary

Once the iterative tree search and bootstrap rectification are complete, IQ-TREE performs a final, highly precise optimization of the model parameters on the overall best tree found.

* **How it functions:** A final, stricter optimization (e.g., `epsilon = 0.010`) is performed to fine-tune the model parameters (like site proportions and rates) for the absolute best-scoring tree topology. This leads to the "Optimal log-likelihood" and "BEST SCORE FOUND" for the entire run (e.g., `-64123.727`).
* **Tree Length:** The "Total tree length" (e.g., `105.079`) is the sum of all branch lengths in the final Maximum Likelihood tree, representing the total amount of evolutionary change inferred.
* **Time Summary:** The log provides a detailed summary of CPU and wall-clock time spent on various stages (ModelFinder, tree search, total), giving insight into the computational cost of the analysis.

* **Key Output Files (Summary):**
    * `my_iqtree_run.iqtree`: The primary, comprehensive report of the entire analysis, including model details, tree statistics, and more.
    * `my_iqtree_run.treefile`: The final Maximum-Likelihood tree, which is the most statistically supported tree topology based on your original data.
    * `my_iqtree_run.log`: The complete log of the entire run, useful for troubleshooting and reviewing all steps.
    * `my_iqtree_run.ckp.gz`: (Checkpoint file) Allows for resuming analysis.
    * `my_iqtree_run.uniqueseq.phy`: (Unique sequences file) Used internally by IQ-TREE.

By following these steps, IQ-TREE efficiently and robustly reconstructs phylogenetic trees, providing not only the most likely evolutionary relationships but also statistical confidence for each inferred branch.